<a href="https://colab.research.google.com/github/summiyazeb/WEB-SEARCH-ENGINE/blob/main/Notebooks/01-Data-Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#setting up the environment
#connect to Drive, create the folder, and get ready to download from Hugging Face.

import os
from google.colab import drive, userdata

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create a folder in your Google Drive for the project
project_dir = '/content/drive/MyDrive/DSA_Search_Engine'
os.makedirs(project_dir, exist_ok=True)
print(f"Folder ready at: {project_dir}")

# 3. Verify that the HF_TOKEN of huggingface, which is in secret key, works
hf_token = userdata.get('HF_TOKEN')
print("Hugging Face token loaded successfully!")

# 4. Install the datasets library
!pip install -q datasets # !: runs terminal command & -q silents output
print("Datasets library installed!")

Mounted at /content/drive
Folder ready at: /content/drive/MyDrive/DSA_Search_Engine
Hugging Face token loaded successfully!
Datasets library installed!


In [ ]:
import os
import re
import json
import html
import statistics
from datasets import load_dataset

# cleaning and loading only the required length dataset

data_url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Video_Games.jsonl"

dataset = load_dataset(
    "json", # tells that its in json format
    data_files=data_url,
    split="train",
    streaming=True, # downloads line by line
    token=hf_token
)

#  function to clean HTML tags and extra spaces
def clean_text(raw_text):
    if not raw_text:
        return ""
    text = html.unescape(raw_text)                 # Convert &quot to "". to normal characters
    text = re.sub(r'<[^>]+>', ' ', text)           # Remove HTML tags like <br /> and converts to space
    text = re.sub(r'\s+', ' ', text).strip()       # Collapse multiple spaces/newlines to one
    return text

# 3. Filter (>= 100 words), clean, and save 55,000 reviews
TARGET_COUNT = 55000 # acc to requirements
MIN_WORDS = 100
output_path = '/content/drive/MyDrive/DSA_Search_Engine/cleaned_dataset.jsonl'

saved_count = 0
word_counts = []

print("Streaming, cleaning, and filtering reviews (minimum 100 words)...")

with open(output_path, 'w', encoding='utf-8') as f:
    for item in dataset:
        cleaned_review = clean_text(item.get('text', ''))
        cleaned_title = clean_text(item.get('title', ''))

        num_words = len(cleaned_review.split()) # count len of each review

        # Keep only reviews with >= 100 words and a valid rating
        if num_words >= MIN_WORDS and item.get('rating') is not None:
            record = {
                "doc_id": saved_count,
                "title": cleaned_title,
                "text": cleaned_review,
                "rating": float(item.get('rating', 0.0)),
                "helpful_vote": int(item.get('helpful_vote', 0)), # use "or 0" instead of ", 0" if reusing on data with None values
                "timestamp": int(item.get('timestamp', 0))
            }
            f.write(json.dumps(record) + '\n')
            word_counts.append(num_words)
            saved_count += 1

            if saved_count % 10000 == 0:
                print(f"Progress: {saved_count:,} / {TARGET_COUNT:,} reviews saved...")

            if saved_count >= TARGET_COUNT: # stops after we got 55,000 reviews
                break

print("\n--- DATASET VERIFICATION SUMMARY ---")
print(f"Total Reviews Saved: {saved_count:,}")
print(f"Minimum Word Count: {min(word_counts)} words")
print(f"Median Word Count: {statistics.median(word_counts)} words")
print(f"Saved permanently at: {output_path}")

Streaming, cleaning, and filtering reviews (minimum 100 words)...
Progress: 10,000 / 55,000 reviews saved...
Progress: 20,000 / 55,000 reviews saved...
Progress: 30,000 / 55,000 reviews saved...
Progress: 40,000 / 55,000 reviews saved...
Progress: 50,000 / 55,000 reviews saved...

--- DATASET VERIFICATION SUMMARY ---
Total Reviews Saved: 55,000
Minimum Word Count: 100 words
Median Word Count: 177.0 words
Saved permanently at: /content/drive/MyDrive/DSA_Search_Engine/cleaned_dataset.jsonl


In [ ]:
import json

output_path = '/content/drive/MyDrive/DSA_Search_Engine/cleaned_dataset.jsonl'

with open(output_path, 'r', encoding='utf-8') as f:
    for i in range(3):
        record = json.loads(f.readline())
        print(f"--- Record {i} ---")
        print(f"ID: {record['doc_id']} | Rating: {record['rating']} | Helpful Votes: {record['helpful_vote']}")
        print(f"Title: {record['title']}")
        print(f"Text Preview: {record['text'][:180]}...\n")

--- Record 0 ---
ID: 0 | Rating: 5.0 | Helpful Votes: 0
Title: if its prime day and you're contemplating, just stop -- do it.
Text Preview: you should probably get these. you'll journal about the next-level experience, or at least sit there and revel for a couple min. honestly, it's like you've never heard footsteps be...

--- Record 1 ---
ID: 1 | Rating: 4.0 | Helpful Votes: 1
Title: Looks good, not sure how functional it is though
Text Preview: See pics, I have a Vive DAS and rubber insulators on the controllers. It does hold everything, and it does seem a bit more organized than just sitting it on the desk... But I'm not...

--- Record 2 ---
ID: 2 | Rating: 5.0 | Helpful Votes: 0
Title: What a great game! Fun for everyone and keeps you moving!
Text Preview: So, I got this and used this game for the first time when I was with my son. He has a WII and his girlfriend (who has two little girls ages 5 and 9) came over with her children whe...

